# 真实数据标签预测（跨工况模型）

本 notebook 用于：
- 加载 `outputs/model_test_cross_condition_260518-2` 下 7 组实验模型；
- 对真实数据特征目录做断丝标签预测；
- 打印正负样本数量、总样本数量、正样本时间；
- 绘制断丝疑似概率（%）随时间变化，并按时间不连续分段绘图。

In [ ]:
from __future__ import annotations

from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ====== 用户配置区 ======
PROJECT_ROOT = Path(r"E:\\codes\\ZZ-BK")
MODEL_ROOT = PROJECT_ROOT / r"outputs\\model_test_cross_condition_260518-2"

# 可配置多个真实特征目录
FEATURE_DIRS = [
    PROJECT_ROOT / r"outputs\\realdata_feature_dataset_20260523",
    PROJECT_ROOT / r"outputs\\realdata_feature_dataset_20260519_v3",
]

# 模型类型：'logistic_regression' | 'rbf_svm' | 'random_forest'
MODEL_TYPE = 'logistic_regression'

# 指定实验组列表，例如 ['exp_01_train_F130', 'exp_07_train_F130_F130A_F130C']
# 设为 None 时自动扫描全部 exp_*
EXPERIMENTS = None

# 二分类阈值：概率 >= 阈值 判为正样本（断丝）
PROB_THRESHOLD = 0.5

# 连续时间分段阈值（秒）
CONTINUITY_GAP_SECONDS = 5.0

OUTPUT_DIR = PROJECT_ROOT / r"outputs\\realdata_prediction_cross_condition"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL_ROOT =', MODEL_ROOT)
print('MODEL_TYPE =', MODEL_TYPE)
print('FEATURE_DIRS =')
for p in FEATURE_DIRS:
    print('  -', p)


In [ ]:
def list_experiments(model_root: Path, experiments: list[str] | None) -> list[str]:
    if experiments:
        return experiments
    return sorted([p.name for p in model_root.iterdir() if p.is_dir() and p.name.startswith('exp_')])

def load_selected_features(exp_dir: Path) -> list[str]:
    sf = exp_dir / 'selected_features.csv'
    if not sf.exists():
        raise FileNotFoundError(f'missing selected_features.csv: {sf}')
    df = pd.read_csv(sf)
    if 'feature' not in df.columns:
        raise ValueError(f"selected_features.csv missing 'feature' column: {sf}")
    features = [str(x) for x in df['feature'].dropna().tolist()]
    if len(features) != 5:
        print(f'[WARN] {exp_dir.name} selected feature count = {len(features)} (expected 5)')
    return features

def load_model(exp_dir: Path, model_type: str):
    model_path = exp_dir / 'models' / f'{model_type}.joblib'
    if not model_path.exists():
        raise FileNotFoundError(f'missing model file: {model_path}')
    model = joblib.load(model_path)
    if not hasattr(model, 'predict_proba'):
        raise TypeError(f'model has no predict_proba: {model_path}')
    return model, model_path

def sorted_feature_files(feature_dir: Path) -> list[Path]:
    return sorted(feature_dir.glob('*window_features*.csv'))

def infer_log_path(feature_csv_path: Path) -> Path | None:
    # 例如：window_features_v3_part_0001.csv -> window_log_v3_part_0001.csv
    cand = feature_csv_path.with_name(feature_csv_path.name.replace('window_features', 'window_log'))
    return cand if cand.exists() else None

def load_feature_with_time(feature_csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(feature_csv_path)
    log_path = infer_log_path(feature_csv_path)
    if log_path is not None:
        log_df = pd.read_csv(log_path)
        if 'window_start_datetime' in log_df.columns:
            if 'window_start_datetime' not in df.columns:
                df['window_start_datetime'] = log_df['window_start_datetime']
            else:
                # 优先保留特征表已有值，缺失时用 log 表补齐
                df['window_start_datetime'] = df['window_start_datetime'].where(
                    df['window_start_datetime'].notna(),
                    log_df['window_start_datetime']
                )
    return df

def build_time_series(df: pd.DataFrame) -> pd.Series:
    # 优先级：window_start_datetime > starttime_raw > arrival_time_raw > offset秒
    for col in ['window_start_datetime', 'starttime_raw', 'arrival_time_raw']:
        if col in df.columns:
            t = pd.to_datetime(df[col], errors='coerce')
            if t.notna().any():
                return t
    offset_s = pd.to_numeric(df.get('window_start_offset_s', np.nan), errors='coerce')
    if np.isfinite(offset_s).any():
        return pd.Timestamp('1970-01-01') + pd.to_timedelta(offset_s.fillna(0.0), unit='s')
    # 最后兜底：按样本序号构造秒级时间
    return pd.to_datetime(pd.RangeIndex(len(df)), unit='s', origin='unix', errors='coerce')

def split_continuous_segments(ts: pd.Series, gap_seconds: float) -> pd.Series:
    # 相邻时间差大于阈值则切分新段
    dt = ts.sort_values().diff().dt.total_seconds()
    cuts = dt.isna() | (dt > gap_seconds)
    return (cuts.cumsum() - 1).rename('segment_id')


In [ ]:
experiments = list_experiments(MODEL_ROOT, EXPERIMENTS)
print('experiments =', experiments)

all_pred_rows = []
summary_rows = []

for exp_name in experiments:
    exp_dir = MODEL_ROOT / exp_name
    selected_features = load_selected_features(exp_dir)
    model, model_path = load_model(exp_dir, MODEL_TYPE)

    print('\n' + '=' * 88)
    print(f'Experiment: {exp_name}')
    print(f'Model     : {MODEL_TYPE} ({model_path.name})')
    print(f'Features  : {selected_features}')

    for feature_dir in FEATURE_DIRS:
        files = sorted_feature_files(feature_dir)
        if not files:
            print(f'[WARN] no feature files in {feature_dir}')
            continue
        print(f'  FeatureDir: {feature_dir.name} files={len(files)}')

        for feature_csv in files:
            df = load_feature_with_time(feature_csv)
            missing = [c for c in selected_features if c not in df.columns]
            if missing:
                print(f'    [SKIP] {feature_csv.name}: missing features -> {missing}')
                continue

            # 核心约束：严格按 selected_features 顺序取列，保证模型输入向量顺序一致
            x = df.loc[:, selected_features].apply(pd.to_numeric, errors='coerce')
            prob_pos = model.predict_proba(x)[:, 1]
            pred = (prob_pos >= PROB_THRESHOLD).astype(int)
            ts = build_time_series(df)

            out = pd.DataFrame({
                'experiment': exp_name,
                'model_type': MODEL_TYPE,
                'feature_dir': feature_dir.name,
                'feature_file': feature_csv.name,
                'source_file_name': df.get('source_file_name', ''),
                'window_id': df.get('window_id', np.arange(len(df))),
                'window_start_offset_s': pd.to_numeric(df.get('window_start_offset_s', np.nan), errors='coerce'),
                'window_start_datetime': ts,
                'prob_pos': prob_pos,
                'prob_pos_pct': prob_pos * 100.0,
                'pred_label': pred,
            })

            n_total = int(len(out))
            n_pos = int((out['pred_label'] == 1).sum())
            n_neg = n_total - n_pos
            print(f'    {feature_csv.name}: pos={n_pos}, neg={n_neg}, total={n_total}')

            summary_rows.append({
                'experiment': exp_name,
                'model_type': MODEL_TYPE,
                'feature_dir': feature_dir.name,
                'feature_file': feature_csv.name,
                'positive_count': n_pos,
                'negative_count': n_neg,
                'total_count': n_total,
                'threshold': PROB_THRESHOLD,
            })
            all_pred_rows.append(out)

if not all_pred_rows:
    raise RuntimeError('No predictions generated. Please check paths and model settings.')

pred_df = pd.concat(all_pred_rows, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
print('\nPrediction dataframe shape =', pred_df.shape)
print('Summary dataframe shape    =', summary_df.shape)


In [ ]:
pred_out_csv = OUTPUT_DIR / f'predictions_{MODEL_TYPE}.csv'
summary_out_csv = OUTPUT_DIR / f'summary_{MODEL_TYPE}.csv'
pred_df.to_csv(pred_out_csv, index=False, encoding='utf-8-sig')
summary_df.to_csv(summary_out_csv, index=False, encoding='utf-8-sig')
print('saved:', pred_out_csv)
print('saved:', summary_out_csv)
summary_df.head(20)


In [ ]:
# 全局统计
global_total = int(len(pred_df))
global_pos = int((pred_df['pred_label'] == 1).sum())
global_neg = global_total - global_pos
print(f'[GLOBAL] pos={global_pos}, neg={global_neg}, total={global_total}, threshold={PROB_THRESHOLD}')

# 分组统计
agg = (
    pred_df.groupby(['experiment', 'feature_dir', 'feature_file'], as_index=False)['pred_label']
    .agg(positive_count=lambda s: int((s == 1).sum()), total_count='count')
)
agg['negative_count'] = agg['total_count'] - agg['positive_count']
agg = agg[['experiment', 'feature_dir', 'feature_file', 'positive_count', 'negative_count', 'total_count']]
agg.sort_values(['experiment', 'feature_dir', 'feature_file']).head(100)


In [ ]:
# 打印正样本时间（断丝样本）
pos_df = pred_df[pred_df['pred_label'] == 1].copy()
pos_df = pos_df.sort_values(['experiment', 'feature_dir', 'feature_file', 'window_start_datetime', 'window_id'])

if pos_df.empty:
    print('当前配置下没有预测为正样本的数据。')
else:
    cols = [
        'experiment', 'feature_dir', 'feature_file', 'source_file_name',
        'window_id', 'window_start_datetime', 'window_start_offset_s', 'prob_pos_pct'
    ]
    print('Positive sample rows =', len(pos_df))
    display(pos_df[cols].head(500))


In [ ]:
# 按连续时间段绘制概率曲线
plot_root = OUTPUT_DIR / f'plots_{MODEL_TYPE}'
plot_root.mkdir(parents=True, exist_ok=True)
plot_count = 0

for (exp_name, feature_dir_name, feature_file), sub in pred_df.groupby(['experiment', 'feature_dir', 'feature_file']):
    sub = sub.copy().sort_values(['window_start_datetime', 'window_id']).reset_index(drop=True)
    ts = pd.to_datetime(sub['window_start_datetime'], errors='coerce')
    valid = ts.notna()
    if not valid.any():
        print(f'[SKIP PLOT] no valid datetime: {exp_name} | {feature_dir_name} | {feature_file}')
        continue

    sub = sub.loc[valid].copy()
    ts = pd.to_datetime(sub['window_start_datetime'])
    sub['segment_id'] = split_continuous_segments(ts, gap_seconds=CONTINUITY_GAP_SECONDS).values

    for segment_id, g in sub.groupby('segment_id'):
        g = g.sort_values('window_start_datetime')
        if len(g) < 2:
            continue

        fig, ax = plt.subplots(figsize=(11, 4))
        x = pd.to_datetime(g['window_start_datetime'])
        y = g['prob_pos_pct']
        ax.plot(x, y, lw=1.4)
        ax.axhline(PROB_THRESHOLD * 100.0, color='r', ls='--', lw=1.0, label=f'threshold={PROB_THRESHOLD:.2f}')
        ax.set_ylim(0, 100)
        ax.set_xlabel('时间 (HH:MM:SS)')
        ax.set_ylabel('断丝疑似概率 (%)')
        ax.set_title(f'{exp_name} | {MODEL_TYPE} | {feature_dir_name} | {feature_file} | seg={int(segment_id)}')
        ax.grid(alpha=0.3)
        ax.legend(loc='upper right')
        fig.autofmt_xdate()

        safe_file = re.sub(r'[^0-9A-Za-z._-]+', '_', feature_file)
        out_png = plot_root / f'{exp_name}__{feature_dir_name}__{safe_file}__seg{int(segment_id):03d}.png'
        fig.savefig(out_png, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        plot_count += 1

print('saved plots =', plot_count)
print('plot dir    =', plot_root)


## 使用说明

1. 在“用户配置区”修改 `FEATURE_DIRS`、`MODEL_TYPE`、`EXPERIMENTS`、`PROB_THRESHOLD`。
2. 顺序执行全部单元格。
3. 输出内容：
- `outputs/realdata_prediction_cross_condition/predictions_<model_type>.csv`
- `outputs/realdata_prediction_cross_condition/summary_<model_type>.csv`
- `outputs/realdata_prediction_cross_condition/plots_<model_type>/*.png`
